In [5]:
# Run this setup only in Google Colab
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Colab Notebooks/ml2_trabalhos_2026")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
except ModuleNotFoundError:
    print("Not running in Colab; skipping Drive mount and install steps.")

Not running in Colab; skipping Drive mount and install steps.


# Trabalho 3 — Marcas d’Água em Dados para Auditoria de Modelos Generativos

> **Pergunta central:** como podemos marcar um conjunto de dados de forma que, se ele for usado no treinamento de um modelo generativo, seja possível auditar posteriormente esse uso?

Neste trabalho usaremos **Variational Autoencoder** como nosso método generativo base, e investigaremos estratégias para **auditoria de uso de dados em modelos generativos**. A situação é a seguinte: o dono de uma base de dados disponibiliza um conjunto de imagens, mas quer preservar a capacidade de verificar, posteriormente, se essas imagens foram usadas por terceiros no treinamento de um modelo generativo.

Para isso, vamos estudar a ideia de inserir uma **marca d’água nos dados de treino**.
Vamos fazer esse estudo usando os dados do MNIST.

Exploraremos duas estratégias:
1. **Marca visível** — patch fixo no canto da imagem. Fácil de detectar, trivialmente removível.
2. **Marca spread-spectrum** — textura pseudo-aleatória de baixa amplitude. Idealmente imperceptível ao olho, detectável via correlação com a chave secreta.

E faremos um **estudo de ablação** variando amplitude, fração marcada e o número de amostras usadas para a auditoria.

**Entregáveis.**
1. Notebook preenchido e executado.
2. Pesos dos VAEs treinados (limpo, marca visível, marca spread-spectrum).
3. Respostas das questões com evidências numéricas dos seus próprios resultados.

**Regras.** Use PyTorch. Fixe seeds. Não apague células do enunciado.

## Parte 0 — Setup e reprodutibilidade

**Tarefa 0.1.** Fixe as seeds aleatórias e configure o dispositivo.

**Tarefa 0.2.** Imprima as versões de `torch` e `torchvision`.

**Tarefa 0.3.** Defina `student_run_tag` (suas iniciais + data) — use ao salvar artefatos.

In [6]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

# adicionais
from tqdm import tqdm
import matplotlib.gridspec as gridspec
from scipy import stats
import plotly.express as px
import pandas as pd

SEED       = 435    # semente do experimentador: splits, pesos, etc.
OWNER_SEED = 1080  # semente do dono dos dados: gera o padrão secreto

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

student_run_tag = "PMRS_2026-26-05"   # ex: "DA_2026-05-22"
output_dir = Path("trabalho3_outputs") / student_run_tag
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {output_dir}")

ModuleNotFoundError: No module named 'plotly'

## Parte 1 — VAE no MNIST

Antes de pensar em marca d'água, precisamos de um modelo generativo razoável. Vamos construir um VAE simples no MNIST.

### 1.1 Dados

**Tarefa 1.1.** Carregue o MNIST de treino e teste com `torchvision.datasets.MNIST` (transformação: apenas `ToTensor()`).

In [ ]:
from torchvision.datasets import MNIST # importando o dataset

transform = transforms.Compose([transforms.ToTensor()])


# definindo os datasets
train_ds = MNIST(root="./data",
                 train=True,
                 transform=transform,
                 download=True
                 )

test_ds  = MNIST(root="./data",
                 train=True,
                 transform=transform,
                 download=True
                 )
# veio com 256
BATCH_SIZE = 256 

# definindo os loaders
train_loader = DataLoader(train_ds,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          drop_last=True,
                          num_workers=32,
                          pin_memory=True,
                          persistent_workers=True,
                          )

test_loader  = DataLoader(train_ds,
                          batch_size=BATCH_SIZE,
                          shuffle=False,
                          drop_last=False,
                          num_workers=32,
                          pin_memory=True,
                          persistent_workers=True,
                          )

print(f"Treino: {len(train_ds):,}  |  Teste: {len(test_ds):,}")

### 1.2 Arquitetura do VAE

Um VAE é composto por:
- **Encoder** $q_\phi(z \mid x)$: projeta a imagem no espaço latente, produzindo $\mu$ e $\log\sigma^2$.
- **Reparametrização**: $z = \mu + \sigma \odot \epsilon$, $\epsilon \sim \mathcal{N}(0, I)$.
- **Decoder** $p_\theta(x \mid z)$: reconstrói a imagem a partir de $z$ (saída em $[0,1]$ via sigmoid).

**Tarefa 1.2.** Implemente `Encoder`, `Decoder`, `VAE`. Use uma MLP simples (camadas escondidas de 512 e 256 unidades com ReLU). A dimensão latente (`LATENT_DIM`) é um hiperparâmetro configurável — para MNIST, valores entre 2 e 32 produzem modelos razoáveis.

In [ ]:
LATENT_DIM = 8   # escolha um valor razoável (sugestão: entre 2 e 32 para MNIST)
IMG_DIM = 28 * 28


class Encoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()

        # TODO: definir self.net (Linear 784→512→256 com ReLU)
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        # TODO: definir self.fc_mu e self.fc_log_var (256 → latent_dim)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.log_var = nn.Linear(256, latent_dim)

    def forward(self, x):

        # TODO: achatar x, passar pela net, retornar (mu, log_var)

        x_flatten = nn.Flatten()(x)
        out = self.net(x_flatten)
        mu = self.fc_mu(out)
        log_var = self.log_var(out)

        return mu, log_var
    

class Decoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()

        # TODO: definir self.net (Linear latent_dim→256→512→784 com ReLU; sigmoid no final)
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid(),
        )

    def forward(self, z):
        # TODO: aplicar self.net e reshape para (-1, 1, 28, 28)

        out = self.net(z)
        out = out.reshape(-1, 1, 28, 28)

        return out


class VAE(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparametrize(self, mu, log_var):

        # TODO: implementar o truque da reparametrização

        std = torch.exp(0.5 * log_var)
        noise = torch.randn_like(std)
        z = mu + std * noise
        
        return z


    def forward(self, x):
        # TODO: encode, sample, decode; retornar (x_hat, mu, log_var)

        mu, log_var = self.encoder(x)
        out = self.reparametrize(mu, log_var)
        x_hat = self.decoder(out)

        return x_hat, mu, log_var


model = VAE(LATENT_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LATENT_DIM = {LATENT_DIM}  |  Parâmetros: {n_params:,}")

### 1.3 Função de perda — ELBO

O VAE maximiza o **Evidence Lower Bound (ELBO)**:

$$\mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x \mid z)]}_{\text{reconstrução}} - \underbrace{D_{KL}(q_\phi(z|x) \,\|\, p(z))}_{\text{regularização}}$$

Para imagens com pixels em $[0,1]$, usamos **BCE** (Bernoulli) como modelo de reconstrução. Como prior, $p(z) = \mathcal{N}(0, I)$, e a KL tem forma fechada:

$$D_{KL}(q_\phi(z|x) \| \mathcal{N}(0,I)) = -\frac{1}{2}\sum_j \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**Tarefa 1.3.** Implemente `vae_loss` que retorna `(loss_total, recon, kl)`. Use BCE com `reduction="sum"` (some sobre pixels e batch).

In [ ]:
def vae_loss(x, x_hat, mu, log_var):
    # TODO: BCE entre x_hat e x (reduction="sum")
    criterion = nn.BCELoss(reduction='sum')
    recon = criterion(x_hat, x)

    # TODO: KL fechada para prior N(0, I)
    var = torch.exp(log_var)
    kl = - 0.5 * torch.sum(1 + log_var - mu**2 - var)

    # TODO: retornar (loss_total, recon, kl)
    loss_total = recon + kl # maximizar o ELBO é o equivalente a minimizar - ELBO,
                            # e recon = Exp[- log p(x | z)] = - Reconstrução

    return loss_total, recon, kl

### 1.4 Treinamento

Treine por **30 épocas** com Adam (lr=1e-3). A loss é a ELBO (somada sobre o batch, dividida no final por `len(dataset)` para reportar por amostra).

**Tarefa 1.4.** Implemente o loop de treino e salve o histórico de loss/recon/KL para treino e teste.

In [ ]:
EPOCHS = 30
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {"train_loss": [], "train_recon": [], "train_kl": [],
           "test_loss":  [], "test_recon":  [], "test_kl":  []}


def run_epoch(loader, train: bool):
    # TODO: alternar entre model.train() / model.eval()
    if train: 
        model.train()
    else:
        model.eval()

    # TODO: acumular loss, recon, kl ao longo do loader
    total_loss = 0.
    total_recon = 0.
    total_kl = 0.

    loop = tqdm(loader, desc="Rodando", leave=False)

    torch.set_grad_enabled(train) # gradiente só ativa quando train = True
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        if train: # se train, zerar o gradiente pra não dar problema
            optimizer.zero_grad()

        outputs, mu, log_var = model(images)
        batch_loss, batch_recon, batch_kl = vae_loss(images, outputs, mu, log_var)

    # TODO: se train, fazer backward + step 
        if train:
            batch_loss.backward()
            optimizer.step()

        total_loss += batch_loss.item()
        total_recon += batch_recon.item()
        total_kl += batch_kl.item()

    # TODO: retornar médias por amostra
    num_samples = len(loader.dataset)
    avg_loss = total_loss / num_samples
    avg_recon = total_recon / num_samples
    avg_kl = total_kl / num_samples

    return avg_loss, avg_recon,avg_kl


for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    te = run_epoch(test_loader,  train=False)
    for key, value in zip(["loss", "recon", "kl"], tr):
        history[f"train_{key}"].append(value)
    for key, value in zip(["loss", "recon", "kl"], te):
        history[f"test_{key}"].append(value)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:3d}/{EPOCHS} | tr loss {tr[0]:.2f}  recon {tr[1]:.2f}  kl {tr[2]:.2f} | te loss {te[0]:.2f}")

torch.save(model.state_dict(), output_dir / "models" / f"vae_clean_latent{LATENT_DIM}.pt") # adicionei a pasta intermediária models
print("Salvo.")

### 1.5 Visualizações

Produza as seguintes visualizações (use o conjunto de teste para a maioria, com seeds fixas).

**Tarefa 1.5a.** Plote as **curvas de treino** (loss total, recon, KL) para treino e teste.

**Tarefa 1.5b.** Mostre **10 reconstruções** lado a lado com os originais.

**Tarefa 1.5c.** Projete o conjunto de teste no espaço latente (use $\mu$, não amostra) e plote um scatter colorido por dígito. *Se `LATENT_DIM > 2`, faça uma redução de dimensionalidade para 2D antes de plotar — use PCA via `numpy.linalg.svd` (centralize os dados antes do SVD; não use sklearn).*

**Tarefa 1.5d.** Amostre 20 imagens do prior ($z \sim \mathcal{N}(0, I)$) e mostre o que o decoder gera.

In [ ]:
# TODO: implementar as 4 visualizações 1.5a–1.5d

# vou fazer em células separadas.

# Salve as figuras em output_dir.
# Serão salvas nas respectivas células

In [ ]:
# - curvas de treinamento (3 subplots)
# 1.5a
fig, axes = plt.subplots(1,3, figsize=(20,5))

# Curva de Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
axes[0].plot(history['test_loss'], label='Test Loss', color='orange', linewidth=2)
axes[0].set_title(f'Curva de Aprendizado (Loss)', fontsize=14)
axes[0].set_xlabel('Épocas', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# Curva de Recon
axes[1].plot(history['train_recon'], label='Train Recon', color='blue', linewidth=2)
axes[1].plot(history['test_recon'], label='Test Recon', color='orange', linewidth=2)
axes[1].set_title(f'Curva de Aprendizado (Recon)', fontsize=14)
axes[1].set_xlabel('Épocas', fontsize=12)
axes[1].set_ylabel('Recon', fontsize=12)
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# Curva de KL
axes[2].plot(history['train_kl'], label='Train KL', color='blue', linewidth=2)
axes[2].plot(history['test_kl'], label='Test KL', color='orange', linewidth=2)
axes[2].set_title(f'Curva de Aprendizado (KL)', fontsize=14)
axes[2].set_xlabel('Épocas', fontsize=12)
axes[2].set_ylabel('KL', fontsize=12)
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(output_dir / "images" / "curvas_1_5.png")
plt.show()

In [ ]:
# - reconstruções (2x10)
# 1.5b
images, labels = next(iter(test_loader)) # iter transforma o test_loader em um iterável e o
                                 # next pega o primeiro mini-batch desse objeto iterável
images_to_show, labels_to_show = images[:10], labels[:10]  # pega as 10 primeiras
images_to_show = images_to_show.to(device)

with torch.no_grad():
    reconstructed, _, _ = model(images_to_show)

fig, axes = plt.subplots(2,10, figsize=(20,5))

# mandando as imagens e as reconstrução pra figura
for i, (img, rec_img) in enumerate(zip(images_to_show, reconstructed)):
    axes[0][i].imshow(img.squeeze().cpu())
    axes[1][i].imshow(rec_img.squeeze().cpu())

# desativando toda a parafernalha que deixa a imagem poluída
for i in range(10):
    axes[0][i].set_xticks([])
    axes[1][i].set_xticks([])
    if i != 0:
        axes[0][i].set_yticks([])
        axes[1][i].set_yticks([])

axes[0][0].set_yticklabels([])
axes[1][0].set_yticklabels([])
axes[0][0].tick_params(left=False)
axes[1][0].tick_params(left=False)

fig.suptitle("10 imagens e suas reconstruções")
axes[0][0].set_ylabel('original', fontsize=12)
axes[1][0].set_ylabel('reconstruído', fontsize=12)
plt.tight_layout()
plt.savefig(output_dir / "images" / "reconstrucoes_1_5.png")
plt.show()

In [ ]:
# - espaço latente em 2D (scatter colorido por dígito; PCA se LATENT_DIM > 2)
#1.5c
all_mu = []
all_labels = []

with torch.no_grad():
    for images, labels in iter(test_loader):
        images = images.to(device)
        batch_mu, _ = model.encoder(images)
        all_mu.append(batch_mu)
        all_labels.append(labels)

all_mu = torch.cat(all_mu).cpu().numpy()
all_labels = torch.cat(all_labels).numpy()

all_mu_centered = all_mu - np.mean(all_mu, axis=0)
_, _, Vh = np.linalg.svd(all_mu_centered, False)

projected = all_mu_centered @ Vh[:2].T

plt.title('Espaço Latente em 2D via PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.scatter(projected[:, 0], projected[:, 1], c = all_labels, cmap = 'tab10', s= 10, alpha=0.8)
plt.colorbar()
plt.savefig(output_dir / "images" / "espaco_latente_1_5.png")
plt.show()

Por curiosidade, resolvi fazer uma visualização extra com PCA em três dimensões. A terceira componente principal não adiciona separação significativa entre as classes. A estrutura esférica e a mistura entre dígitos se mantém semelhante à projeção 2D.

In [ ]:
# - espaço latente em 3D (scatter colorido por dígito; PCA se LATENT_DIM > 3)
#1.5c - parte 2

projected_3d = all_mu_centered @ Vh[:3].T

import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'x': projected_3d[:, 0],
    'y': projected_3d[:, 1],
    'z': projected_3d[:, 2],
    'dígito': all_labels.astype(str)
}).sample(10000, random_state=SEED) # pra não ficar muito pesado, vou amostrar 10000 amostras.

fig = px.scatter_3d(df, x='x', y='y', z='z', color='dígito',
                    title='Espaço Latente em 3D via PCA',
                    opacity=1,
                    width=1000, height=800)
fig.update_traces(marker=dict(size=4))
fig.update_layout(legend=dict(itemsizing='constant'))
fig.update_traces(marker=dict(size=4))
fig.show()

(output_dir / "html").mkdir(parents=True, exist_ok=True)
fig.write_html(str(output_dir / "html" / "latente_3d.html"))
fig.write_image(str(output_dir / "images" / "latente_3d.png"))

# observação: se você clicar em um dígito na legenda à direita, você

In [ ]:
# imagem exibida pra quando não tiver acesso a visualização interativa
from IPython.display import Image
Image(str(output_dir / "images" / "latente_3d.png"))

In [ ]:
# - amostras do prior (2x10)
# 1.5d
z = torch.randn(20, LATENT_DIM).to(device) # amostras
with torch.no_grad():
    decoded_z = model.decoder(z) # amostras passadas pelo decoder

fig, axes = plt.subplots(2,10, figsize=(20,5))

# mandando as imagens e as reconstrução pra figura
for i, img in enumerate(decoded_z[:10]):
    axes[0][i].imshow(img.squeeze().cpu())

for i, img in enumerate(decoded_z[10:]):
    axes[1][i].imshow(img.squeeze().cpu())

# desativando toda a parafernalha que deixa a imagem poluída
for i in range(10):
    axes[0][i].axis("off")
    axes[1][i].axis("off")

fig.suptitle("output do decoder para imagens da prior")
plt.tight_layout()
plt.savefig(output_dir / "images" / "decoder_em_amostras_da_prior_1_5.png")
plt.show()

### Questão 1 — Análise do VAE base

**a)** Compare as curvas de treino e teste. O modelo apresenta underfitting, overfitting, ou está bem ajustado? Justifique com o gap treino/teste e a tendência das curvas.

**b)** Compare reconstruções com amostras do prior. Em qual das duas o VAE produz imagens melhores? Por quê?

**c)** Na projeção 2D do espaço latente (direta se `LATENT_DIM=2`, via PCA caso contrário), as classes ficam separadas? Existem regiões vazias? Conecte a estrutura latente à qualidade das amostras geradas.

*Escreva sua resposta aqui (a, b, c).*

**a)** O modelo parece bem ajustado. O gap treino/teste é pequeno nas três curvas e todas indicam tendência de estabilização.

**b)** As imagens produzidas pela reconstrução são, em geral, melhores do que as amostras da prior. As reconstruções são melhores pois o decoder recebe o z proveniente de uma imagem real. Na prática, o encoder manda o z para uma região que o decoder conhece bem. Já nas amostras do prior, o z é aleatório e pode cair em uma região que o decoder nunca viu durante o treino.

**c)**  Apesar de ser possível notar uma prevalência de certas classes em certas regiões do plano, as mesmas ainda estão misturadas. Com a perda de 6 dimensões na projeção PCA, a separação que existe no espaço original não fica claramente visível em 2D.
É possível ver as classes se concentrando em uma esfera de raio aproximadamente 3, e há poucas regiões vazias dentro dessa esfera, nenhuma no centro, mas todo o espaço fora dessa esfera é vazio.
Note que a região vazia exterior à esfera na visualização é a parte que o decoder não entende bem. Amostras de prior que caem nessas regiões vazias tendem a gerar imagens de qualidade inferior.
Podemos observar algumas coisas interessantes. As classes 0 e 1 tem símbolos muito diferentes e estão localizadas de forma diametralmente oposta na visualização.Ademais, veja que a classe 1 está isolada no lado direito do gráfico, com pouquíssima sobreposição com outras classes. Isso indica que, uma vez que o 1 tem uma estrutura muito simples e particular, o encoder manda todas as imagens de 1 para o mesmo espaço. **MODIFICAR RESPOSTA**


#### Experimento: decodificando ao longo do espaço latente

Como visto na visualização anterior, espaço latente aprendido pelo VAE concentra as representações dos dígitos dentro de uma esfera de raio aproximadamente 3 no plano PCA. O que acontece quando percorremos esse espaço, tanto dentro quanto fora dessa esfera?

Vamos percorrer alguns segmentos do plano 2D do PCA. Vamos converter alguns pontos do segmento para o espaço latente 8D via projeção inversa e decodificamos as imagens resultantes. Os experimentos cobrem as diagonais, eixos horizontais e verticais, e bordas da esfera.

In [ ]:
def plot_line(n, x_range, y_range, filename):

    x = np.linspace(*x_range, n)
    y = np.linspace(*y_range, n)
    points_2d = np.stack([x, y], axis=1)

    points_8d = points_2d @ Vh[:2]
    points_8d = points_8d + np.mean(all_mu, axis=0)
    points_8d = torch.tensor(points_8d, dtype=torch.float32).to(device)

    gen_images = []
    with torch.no_grad():
        for point in points_8d:
            decoded_point = model.decoder(point.unsqueeze(0))
            gen_images.append(decoded_point.squeeze().cpu())

    fig = plt.figure(figsize=(20, 5))
    gs_outer = gridspec.GridSpec(1, 2, width_ratios=[2, 5], wspace=0.05)

    ax_scatter = fig.add_subplot(gs_outer[0])
    ax_scatter.scatter(projected[:, 0], projected[:, 1], c=all_labels, cmap='tab10', s=10, alpha=0.4)
    ax_scatter.plot([x_range[0], x_range[1]], [y_range[0], y_range[1]], color='black', linewidth=2)
    ax_scatter.set_aspect('equal')
    ax_scatter.set_xticks([])
    ax_scatter.set_yticks([])
    ax_scatter.set_title("mapa", fontsize=8)

    gs_inner = gridspec.GridSpecFromSubplotSpec(5, 15, subplot_spec=gs_outer[1], wspace=0, hspace=0.4)

    for i, image in enumerate(gen_images):
        ax = fig.add_subplot(gs_inner[i//15, i%15])
        ax.imshow(image)
        ax.set_title(f"({x[i]:.1f}, {y[i]:.1f})", fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(f"percorrendo ({x_range[0]:.1f}, {y_range[0]:.1f}) → ({x_range[1]:.1f}, {y_range[1]:.1f}) - {filename}")
    plt.savefig(output_dir / "images" / f"{filename}.png")
    plt.show()

def section(title):
    print(f"\n{'─' * 20} {title.upper()} {'─' * 20}\n")

In [ ]:
section('eixos x e y puros')
plot_line(75, (-4, 4), (0, 0), "horizontal")
plot_line(75, (0, 0), (-4, 4), "vertical")

section('zoom nos eixos puros')
plot_line(75, (-1, 1), (0, 0), "horizontal_zoom")
plot_line(75, (0, 0), (-1, 1), "vertical_zoom")
plot_line(75, (-0.5, 0.5), (0, 0), "horizontal_superzoom")
plot_line(75, (0, 0), (-0.5, 0.5), "vertical_superzoom")

section('explorando diagonal principal')
plot_line(75, (-4, 4), (-4, 4), "diagonal")
plot_line(75, (-2, 2), (-2, 2), "diagonal_concentrada")
plot_line(75, (-1, 1), (-1, 1), "zoom_transicao")
plot_line(75, (-0.5, 0.5), (-0.5, 0.5), "zoom_origem")

section('explorando diagonal inversa')
plot_line(75, (-4, 4), (4, -4), "diagonal_inversa")
plot_line(75, (-2, 2), (2, -2), "diagonal_inversa_concentrada")
plot_line(75, (-1, 1), (1, -1), "zoom_transicao_invertida")
plot_line(75, (-0.5, 0.5), (0.5, -0.5), "zoom_origem_invertida")

section('partes horizontal e bordas')
plot_line(75, (-4, 4), (2, 2), "horizontal_cima")
plot_line(75, (-4, 4), (-2, -2), "horizontal_baixo")
plot_line(75, (-4, -4), (-4, 4), "borda_esquerda")
plot_line(75, (4, 4), (-4, 4), "borda_direita")

section('diagonais internas nos quadrantes')
plot_line(75, (-4, 0), (0, 4), "diagonal_superior_esquerda")
plot_line(75, (0, 4), (4, 0), "quadrante_superior_esquerda")

section('horizontais acima e abaixo da esfera')
plot_line(75, (-4, 4), (4, 4), "topo")
plot_line(75, (-4, 4), (-4, -4), "base")                

#### Comentário do Experimento

**TEM QUE PREENCHER**

## Parte 2 — Marca d'água visível (warmup do auditor)

A versão mais simples do problema: o dono dos dados estampa um **patch branco fixo** num canto das imagens. Esta marca é trivialmente visível, mas serve como referência para entendermos o "limite superior" da detectabilidade.

### 2.1 Definir a marca e o dataset

A marca é um quadrado branco $4 \times 4$ no canto inferior direito (posição `[22:26, 22:26]`), valor $1.0$. Marcamos 20% das imagens de treino, selecionadas aleatoriamente com seed fixa.

**Tarefa 2.1.** Implemente:
- A função `apply_visible_watermark(img)`.
- A classe `VisibleWatermarkedMNIST` que envolve `train_ds` e marca aleatoriamente uma fração das imagens, mantendo `watermarked_indices` para análise posterior.

In [ ]:
PATCH_ROW, PATCH_COL = 22, 22
PATCH_SIZE = 4
WM_FRACTION = 0.20


def apply_visible_watermark(img: torch.Tensor) -> torch.Tensor:
    # TODO: clone e estampe um quadrado branco PATCH_SIZE x PATCH_SIZE
    wtmrk_img = img.detach().clone()
    wtmrk_img[:, PATCH_ROW:PATCH_ROW + PATCH_SIZE, PATCH_COL:PATCH_COL + PATCH_SIZE] = 1.0
    return wtmrk_img

class VisibleWatermarkedMNIST(Dataset):
    def __init__(self, base_dataset, fraction=WM_FRACTION, seed=SEED):
        # TODO: amostrar índices a marcar (use np.random.default_rng(seed))
        self.base_dataset = base_dataset
        rng = np.random.default_rng(seed)
        self.watermarked_indices  = set(rng.choice(len(self.base_dataset), int(fraction * len(self.base_dataset)), replace=False))

    def __len__(self):
        return len(self.base_dataset)
        
    def __getitem__(self, idx):
        # TODO: retornar (img marcada se idx in indices else img, label)
        img, label = self.base_dataset[idx]
        if idx in self.watermarked_indices:
            return apply_visible_watermark(img), label
        else:
            return img, label


vis_ds = VisibleWatermarkedMNIST(train_ds)
print(f"Marcadas: {len(vis_ds.watermarked_indices):,} / {len(vis_ds):,}")

In [ ]:
# TODO: mostre 10 amostras originais e suas versões marcadas (2x10 grid)

rng = np.random.default_rng(SEED) # fixando o gerador
wtrmk_indices = [int(i) for i in vis_ds.watermarked_indices] # coletando os índices das imagens marcadas
to_show_indices = rng.choice(wtrmk_indices, 10, replace=False) # escolhendo 10 imagens marcadas para mostrar
print(f"índice das imagens a serem mostradas:\n {to_show_indices}") # índices das imagens a serem mostradas

not_watermarked_imgs = []
watermarked_imgs = []

for idx in to_show_indices:
    not_watermarked_imgs.append(train_ds[idx][0])
    watermarked_imgs.append(vis_ds[idx][0])

fig, axes = plt.subplots(2,10, figsize=(20,5))

# mandando as imagens e e suas versões marcadas pra figura
for i, (img, marked_img) in enumerate(zip(not_watermarked_imgs, watermarked_imgs)):
    axes[0][i].imshow(img.squeeze().cpu())
    axes[1][i].imshow(marked_img.squeeze().cpu())

# desativando toda a parafernalha que deixa a imagem poluída
for i in range(10):
    axes[0][i].set_xticks([])
    axes[1][i].set_xticks([])
    if i != 0:
        axes[0][i].set_yticks([])
        axes[1][i].set_yticks([])

axes[0][0].set_yticklabels([])
axes[1][0].set_yticklabels([])
axes[0][0].tick_params(left=False)
axes[1][0].tick_params(left=False)

fig.suptitle("10 imagens e suas versões marcadas")
axes[0][0].set_ylabel('original', fontsize=12)
axes[1][0].set_ylabel('marcadas', fontsize=12)
plt.tight_layout()
plt.savefig(output_dir / "images" / "marcacoes_simples_2_1.png")
plt.show()

### 2.2 Treinar o VAE marcado

Treine um novo VAE (`vae_vis`) com **a mesma arquitetura** do `model` (mesma `LATENT_DIM`) mas usando `vis_ds` como dataset de treino. Use 30 épocas, Adam lr=1e-3.

**Dica.** Modularize: escreva uma função `train_vae(loader, latent_dim, epochs, label)` que cria, treina e retorna um novo VAE. Vamos reutilizá-la várias vezes nas próximas partes.

In [ ]:
# vamos antes definir uma função de rodar épocas mais flexível

def best_run_epoch(loader, model, train: bool, optimizer=None):

    if train: 
        model.train()
    else:
        model.eval()

    total_loss = 0.
    total_recon = 0.
    total_kl = 0.

    loop = tqdm(loader, desc="Rodando" if train else "Testando", leave=False)

    torch.set_grad_enabled(train)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        if train and optimizer is not None:
            optimizer.zero_grad()

        outputs, mu, log_var = model(images)
        batch_loss, batch_recon, batch_kl = vae_loss(images, outputs, mu, log_var)

        if train and optimizer is not None:
            batch_loss.backward()
            optimizer.step()

        total_loss += batch_loss.item()
        total_recon += batch_recon.item()
        total_kl += batch_kl.item()

    num_samples = len(loader.dataset)
    avg_loss = total_loss / num_samples
    avg_recon = total_recon / num_samples
    avg_kl = total_kl / num_samples

    return avg_loss, avg_recon,avg_kl

def train_vae(loader, latent_dim, epochs, label, N=5, lr=1e-3):
    # TODO: criar VAE(latent_dim), Adam, loop de épocas
    model = VAE(latent_dim).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_recon": [], "train_kl": [],
            "test_loss":  [], "test_recon":  [], "test_kl":  []}

    for epoch in range(1, epochs + 1):
        tr = best_run_epoch(loader, model, train=True, optimizer=optimizer)
        te = best_run_epoch(test_loader,  model, train=False)
        for key, value in zip(["loss", "recon", "kl"], tr):
            history[f"train_{key}"].append(value)
        for key, value in zip(["loss", "recon", "kl"], te):
            history[f"test_{key}"].append(value)
        if epoch % N == 0 or epoch == 1:
            print(f"Ep {epoch:3d}/{epochs} | tr loss {tr[0]:.2f}  recon {tr[1]:.2f}  kl {tr[2]:.2f} | te loss {te[0]:.2f}")

    # TODO: print resumo a cada N épocas
    model_path = output_dir / "models" / f"vae_{label}.pt"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_path)
    print(f"Modelo salvo em: {model_path}")

    return model


# vis_loader = DataLoader(vis_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

# vou usar um loader um pouco modificado
vis_loader = DataLoader(vis_ds,
                        batch_size=BATCH_SIZE,
                        shuffle=True,
                        drop_last=True,
                        num_workers=32,
                        pin_memory=True,
                        persistent_workers=True,
                        )
# tive que recriar o test loader pq tava dando pau
test_loader = DataLoader(test_ds, 
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=32,
                         pin_memory=True,
                         persistent_workers=True
                         )

vae_vis = train_vae(vis_loader, LATENT_DIM, epochs=30, label="vis")
# torch.save(vae_vis.state_dict(), output_dir / "vae_visible.pt")
# comentei a linha de cima pois a própria função já salva o cara como vae_vis

### 2.3 Auditoria visual

Use **o mesmo $z$** para gerar amostras de `model` (limpo) e `vae_vis` (com marca). Mostre lado a lado.

**Tarefa 2.3.** Plote 20 amostras de cada modelo (mesmo $z$), em duas linhas — limpo em cima, marcado embaixo.

In [ ]:
N = 20
# TODO: amostre o mesmo z, decodifique nos 2 modelos, plote lado a lado

z = torch.randn(N, LATENT_DIM).to(device)

with torch.no_grad():
    vae_decoded_z = model.decoder(z)
    vis_decoded_z = vae_vis.decoder(z)

fig, axes = plt.subplots(2,N, figsize=(20,3))

# mandando as imagens e as reconstrução pra figura
for i, (img, vis_img) in enumerate(zip(vae_decoded_z, vis_decoded_z)):
    axes[0][i].imshow(img.squeeze().cpu())
    axes[1][i].imshow(vis_img.squeeze().cpu())

# desativando toda a parafernalha que deixa a imagem poluída
for i in range(N):
    axes[0][i].set_xticks([])
    axes[1][i].set_xticks([])
    if i != 0:
        axes[0][i].set_yticks([])
        axes[1][i].set_yticks([])

axes[0][0].set_yticklabels([])
axes[1][0].set_yticklabels([])
axes[0][0].tick_params(left=False)
axes[1][0].tick_params(left=False)

fig.suptitle("amostras do prior: VAE limpo vs VAE com marca visível (mesmo z)")
axes[0][0].set_ylabel('sem marca', fontsize=12)
axes[1][0].set_ylabel('com marca', fontsize=12)
plt.tight_layout()
plt.savefig(output_dir / "images" / "auditoria_visual_2_3.png", dpi=120, bbox_inches='tight')
plt.show()

### 2.4 Auditoria estatística

Quantifique o sinal calculando o **valor médio de pixel** dentro da região do patch:

$$\text{patch\_mean}(x) = \frac{1}{|P|} \sum_{(i,j) \in P} x_{ij}, \quad P = \{(i,j) : 22 \le i, j < 26\}$$

**Tarefa 2.4.** Implemente `patch_mean(imgs)` (tensor `(N, 1, 28, 28)` → tensor `(N,)`). Compute essa métrica para 4 grupos e plote um bar chart comparativo com erro padrão:
1. Imagens limpas do teste (primeiras 1000).
2. Imagens marcadas do treino (primeiros 1000 índices marcados).
3. 1000 amostras do `model` (VAE limpo).
4. 1000 amostras do `vae_vis`.

In [ ]:
def patch_mean(imgs: torch.Tensor) -> torch.Tensor:
    # TODO: retornar média de pixels na região do patch, por imagem
    region = imgs[:, :, 22:26, 22:26]
    region_mean = region.mean(dim=(1,2,3))
    
    return region_mean

N_AUDIT = 1000

# TODO: computar patch_mean para os 4 grupos, plotar bar chart, calcular lift

# definindo os grupos
grp1_imgs = torch.stack([test_ds[i][0] for i in range(N_AUDIT)]).to(device)
grp2_imgs = torch.stack([vis_ds[i][0] for i in range(N_AUDIT)]).to(device)

with torch.no_grad():
    z_clean = torch.randn(N_AUDIT, LATENT_DIM).to(device)
    grp3_imgs = model.decoder(z_clean)

with torch.no_grad():
    z_vis = torch.randn(N_AUDIT, LATENT_DIM).to(device)
    grp4_imgs = vae_vis.decoder(z_vis)

# computando patch_mean pros grupos e salvando num dicionário
groups = {
    "teste limpo": patch_mean(grp1_imgs).cpu().numpy(),
    "treino marcado": patch_mean(grp2_imgs).cpu().numpy(),
    "VAE limpo (gerado)": patch_mean(grp3_imgs).cpu().numpy(),
    "VAE Vis (gerado)": patch_mean(grp4_imgs).cpu().numpy(),
}

groups_names = list(groups.keys())

# computando as estatísticas pra cada grupo
means = [] # média
sem = [] # erro padrão, indica a precisão da estimatvia da média

for name, values in groups.items():
    means.append(np.mean(values))
    sem.append(stats.sem(values))

# plotando o bar_chart
plt.figure(figsize=(10, 6))
bars= plt.bar(groups_names, means, yerr=sem, capsize=8, color = ["#A9D2DA", "#E8A1A8", "#79E6BB", "#C6B5F2"], edgecolor="black")

plt.errorbar([], [], yerr=[], fmt=' ', ecolor='black', capsize=4, label="Barras pretas: Margem de erro e certeza da média")
plt.ylabel("Valor Médio do Pixel no Patch")
plt.xlabel("Auditoria Estatística: Média de Pixel na Região do Patch (22:26)")
plt.grid(axis='y', linestyle="--", alpha=0.7)
plt.legend(loc="upper left", fontsize=9, framealpha=0.9, edgecolor="gray")


for bar, error in zip(bars, sem):
    yval = bar.get_height()
    altura_texto = yval + error + 0.005
    
    plt.text(
        bar.get_x() + bar.get_width()/2.0, 
        altura_texto, 
        f"{yval:.4f}", 
        ha='center', 
        va='bottom', 
        fontweight='bold',
        fontsize=8
    )
plt.show()

lift_data = means[1] - means[0]    # treino marcado - teste limpo
lift_models = means[3] - means[2]  # VAE Vis - VAE Limpo

print(f"Lift nos Dados Reais (Sinal do Patch): {lift_data:.4f}")
print(f"Lift nos Modelos Gerados (Memorização): {lift_models:.4f}")

### Questão 2 — Auditoria da marca visível

**a)** Nas amostras lado a lado (Tarefa 2.3), o patch aparece nas amostras do VAE marcado? Em quantas das 20 é visualmente óbvio?

**b)** Compare `patch_mean` das amostras geradas pelos dois VAEs. Reporte os valores e o lift. O que esse número significa fisicamente?

**c)** Proponha um ataque concreto que o dono do modelo poderia aplicar para destruir este sinal sem prejudicar muito a qualidade visual.

*Escreva suas respostas aqui (a, b, c).*

## Parte 3 — Marca d'água imperceptível (spread-spectrum)

A marca visível é trivial de remover. Agora projetamos uma marca **invisível ao olho** mas **detectável estatisticamente** — a base dos sistemas de watermarking spread-spectrum.

### 3.1 O padrão secreto

A marca usa uma textura pseudo-aleatória $w \in \mathbb{R}^{28 \times 28}$ conhecida apenas pelo auditor (o dono dos dados). A imagem marcada é:

$$x' = \text{clamp}(x + \varepsilon \cdot w,\, 0, 1)$$

O padrão é gerado a partir de `OWNER_SEED` — apenas o dono dos dados conhece essa chave e pode regenerar $w$ para auditar. O treinador do modelo não precisa conhecê-la.

**Tarefa 3.1.** Execute a célula abaixo — ela já está pronta. O código gera `wm_pattern` a partir de `OWNER_SEED` (usando `torch.Generator` para não perturbar a semente global), **subtrai a média** e normaliza pelo valor absoluto máximo (cada elemento fica em $[-1, 1]$). A visualização usa o mapa de cor `RdBu`. Confirme que a média impressa é $\approx 0$ e que $\max|w| = 1$.

In [ ]:
EPSILON        = 0.10
WM_FRACTION_SS = 0.20

# Gerado com OWNER_SEED (isolado do RNG global)
_owner_gen = torch.Generator().manual_seed(OWNER_SEED)
wm_pattern = torch.randn(1, 28, 28, generator=_owner_gen)
wm_pattern = wm_pattern - wm_pattern.mean()
wm_pattern = wm_pattern / wm_pattern.abs().max()

print(f"wm_pattern — média: {wm_pattern.mean().item():.2e}  max|w|: {wm_pattern.abs().max().item():.2f}")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(wm_pattern.squeeze(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_title("Padrão secreto $w$"); ax.axis("off")
plt.tight_layout(); plt.savefig(output_dir / "ss_pattern.png", dpi=120); plt.show()

### 3.2 Dataset com marca aditiva

**Tarefa 3.2.** Implemente `SpreadSpectrumMNIST` no mesmo padrão de `VisibleWatermarkedMNIST`, mas aplicando `x' = clamp(x + EPSILON * wm_pattern, 0, 1)`. Visualize um exemplo: original, marcado, e a diferença (amplificada para ser visível).

In [ ]:
def apply_ss_watermark(img):
    # TODO
    raise NotImplementedError


class SpreadSpectrumMNIST(Dataset):
    # TODO
    pass


ss_ds = SpreadSpectrumMNIST(train_ds, fraction=WM_FRACTION_SS)
print(f"Marcadas: {len(ss_ds.wm_indices):,} / {len(ss_ds):,}")

# TODO: visualizar original vs marcado vs (marcado-original)*5

### 3.3 Treinar VAE com a marca imperceptível

Use `train_vae` (definida na Parte 2) para treinar `vae_ss` em `ss_ds`. Mesma arquitetura, 30 épocas.

In [ ]:
ss_loader = DataLoader(ss_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
# TODO: treine vae_ss e salve
raise NotImplementedError

### 3.4 Detector ingênuo de correlação (e por que ele tem um problema)

O detector natural é a **correlação** entre uma imagem gerada e o padrão secreto:

$$\text{score}(x) = \langle x,\, w \rangle = \sum_{i,j} x_{ij} \, w_{ij}$$

**Tarefa 3.4a.** Implemente `corr_score(imgs, pattern)` e calcule os scores para 2000 amostras do `model` (limpo) e 2000 do `vae_ss` (marcado). Plote ambos como histogramas sobrepostos.

In [ ]:
def corr_score(imgs, pattern):
    # TODO: produto interno por imagem
    raise NotImplementedError


N_AUDIT = 2000
model.eval(); vae_ss.eval()
# TODO: gerar amostras com o mesmo z, computar scores, plotar histogramas
raise NotImplementedError

### 3.5 Diagnóstico: por que o VAE limpo não está centrado em zero?

Você provavelmente observou que o histograma do **VAE limpo** não está centrado em $0$. Isso parece contraditório: o modelo nunca viu $w$, então a correlação não deveria ser ruído puro?

**Tarefa 3.5.** Explique matematicamente por que $\mathbb{E}[\langle x_{\text{clean}}, w \rangle] \ne 0$ em geral. Calcule numericamente o valor que causa esse desvio e mostre que ele coincide com a média empírica do histograma.

In [ ]:
# TODO: compute a imagem média do MNIST limpo (mu_mnist) e seu produto interno com wm_pattern
# Compare com a média do histograma do VAE limpo da seção anterior — devem ser parecidos.
raise NotImplementedError

### 3.6 Detector centralizado

Para o detector ser interpretável (histograma centrado em zero sob $H_0$), subtraia o offset $\langle \mu_{\text{MNIST}}, w \rangle$ centralizando as imagens antes de calcular a correlação:

$$\text{corr\_centered}(x) = \langle x - \mu_{\text{MNIST}},\, w \rangle$$

**Tarefa 3.6.** Implemente `corr_score_centered(imgs, pattern, mean_img)` e refaça o histograma. O histograma do VAE limpo deve agora aparecer próximo de zero.

In [ ]:
def corr_score_centered(imgs, pattern, mean_img):
    # TODO: <x - mean_img, pattern>
    raise NotImplementedError


# TODO: recomputar os scores e replotar o histograma centrado
raise NotImplementedError

### 3.7 Teste de hipótese

Com os scores centralizados, formalizamos a auditoria como um **teste de duas amostras**. O auditor gera $N$ amostras independentes do modelo suspeito e $N$ amostras independentes de um modelo de referência limpo, e testa se as médias diferem:

$$t = \frac{\bar{s}_{\text{ss}} - \bar{s}_{\text{clean}}}{\sqrt{(\hat{\sigma}_{\text{ss}}^2 + \hat{\sigma}_{\text{clean}}^2)/N}}$$

**Tarefa 3.7.** Gere 2000 amostras independentes de cada modelo (use `torch.randn` separado para cada). Compute `sc_clean`, `sc_ss` e o t-estatístico acima. Plote os histogramas e imprima o t-estatístico.

In [ ]:
N_AUDIT = 2000

# TODO: gerar N_AUDIT amostras INDEPENDENTES de cada modelo
# (dois torch.randn separados — não compartilhe z entre os modelos)
# Compute sc_clean e sc_ss usando corr_score_centered
# Compute o t-estatístico de duas amostras
raise NotImplementedError

### Questão 3 — Análise do detector spread-spectrum

**a)** Descreva o histograma ingênuo (Tarefa 3.4): os dois grupos estão centrados onde você esperaria? Você conseguiria detectar a marca sem ajustes?

**b)** Explique matematicamente por que o VAE limpo não está centrado em zero. Reporte o valor de $\langle \mu_\text{MNIST}, w \rangle$ que você calculou e compare com a média observada no histograma.

**c)** Após os fixes (Tarefa 3.6), os histogramas estão centrados e separados? Reporte o t-estatístico e interprete: a marca é detectável com N=2000?

*Escreva suas respostas aqui (a, b, c).*

## Parte 4 — Estudo controlado: quanto sinal e quantos dados?

A Parte 3 usou uma única configuração ($\varepsilon = 0.10$, fração $= 0.20$). Aqui mapeamos sistematicamente como a detectabilidade depende dessas escolhas.

> *Dica: você pode usar menos épocas por VAE na varredura para reduzir o tempo de execução. Se fizer isso, retreine o VAE limpo com o mesmo número de épocas antes da varredura — a comparação só é justa quando todos os modelos treinam sob as mesmas condições.*

### 4.1 Fábrica de datasets

**Tarefa 4.1.** Implemente `make_ss_dataset(eps, fraction)` que cria um dataset com `fraction` das imagens marcadas com amplitude `eps`.

In [ ]:
# TODO: escolha pelo menos 2 valores para cada parâmetro
EPSILON_VALUES  = [...]   # ex: [0.05, 0.10]
FRACTION_VALUES = [...]   # ex: [0.10, 0.20]
N_AUDIT         = 2000    # fixo
EPOCHS_ABLATION = ...     # pode ser menor que EPOCHS — veja a dica acima

def make_ss_dataset(eps, fraction, seed=SEED):
    # TODO: retornar um Dataset que aplica a marca em `fraction` das imagens de treino
    # com amplitude `eps`
    raise NotImplementedError

# Retreina o VAE limpo com EPOCHS_ABLATION para comparação justa
# TODO: chamar train_vae com train_loader, LATENT_DIM, EPOCHS_ABLATION
model_ab = ...
raise NotImplementedError

### 4.2 Varredura

**Tarefa 4.2.** Execute a varredura sobre a grade `EPSILON_VALUES × FRACTION_VALUES` (o `model_ab` já foi retreinado na 4.1 acima):
1. Para cada (ε, fração), treine um VAE marcado com `EPOCHS_ABLATION` épocas.
2. Para cada par, gere `N_AUDIT` amostras **independentes** de `model_ab` e do VAE marcado.
3. Compute o t-estatístico de duas amostras (mesma fórmula da Parte 3.7) e registre o sinal médio.

Guarde os resultados numa lista de dicts com chaves `eps`, `frac`, `signal`, `t`.

In [ ]:
# TODO: implementar a varredura
# Para cada (eps, frac) em EPSILON_VALUES × FRACTION_VALUES:
#   1. Crie o dataset com make_ss_dataset(eps, frac)
#   2. Treine um VAE com EPOCHS_ABLATION épocas
#   3. Gere N_AUDIT amostras INDEPENDENTES de model_ab e do VAE marcado
#   4. Compute o t-estatístico de duas amostras e o sinal médio
#   5. Guarde em results como dict {"eps": ..., "frac": ..., "signal": ..., "t": ...}
results = []
raise NotImplementedError

### 4.3 Visualização dos resultados

**Tarefa 4.3.** Plote dois heatmaps com eixos $(\varepsilon, \text{fração})$:
1. Sinal médio $\bar{d} = \bar{s}_{\text{wm}} - \bar{s}_{\text{clean}}$.
2. T-estatístico (duas amostras independentes).

Anote o valor em cada célula do heatmap.

In [ ]:
# TODO: heatmap signal_grid, heatmap t_grid
raise NotImplementedError

### Questão 4 — Análise dos resultados da ablação

**a)** Olhando o heatmap de sinal médio, como $\bar{d}$ cresce com $\varepsilon$ e com a fração? O crescimento é aproximadamente linear em cada eixo? Use 2–3 pares de células para justificar com números.

**b)** No heatmap de t-estatístico, qual é a menor combinação $(\varepsilon, \text{fração})$ que atinge $|t| > 3$? Existem células onde a marca não foi detectável mesmo com N=2000?

**c)** Com base nos seus resultados, qual é o par $(\varepsilon, \text{fração})$ mínimo que você recomendaria para um cenário de auditoria real? Justifique considerando tanto a detectabilidade quanto o custo de marcar os dados.

*Escreva suas respostas aqui (a, b, c).*

## Parte 5 — Extensões (Bônus)

Esta parte é **opcional**. **Escolha *uma* das duas questões abaixo** e implemente-a. Isso rende +10% na nota final.

### 5.1 — Robustez da marca a um ataque defensivo

Suponha que o dono do modelo gerador, sabendo que pode existir uma marca spread-spectrum, aplica um filtro às amostras antes de liberá-las. Implemente o ataque mais simples possível: **blur Gaussiano** nas amostras do `vae_ss` antes de calcular o score.

Rode o detector centralizado da Parte 3 em três condições: (i) sem ataque, (ii) blur com $\sigma=0.5$, (iii) blur com $\sigma=1.0$. Use $\varepsilon=0.10$, frac $=0.20$, $N=2000$.

Reporte os três valores de $t$ numa tabela e responda: o sinal sobrevive? A partir de que $\sigma$ a marca se torna indetectável ($|t| < 3$)?

*Escreva sua resposta aqui (tabela + análise).*

### 5.2 — Efeito da capacidade do canal latente

Treine **um VAE adicional** com uma `LATENT_DIM` diferente da sua escolha original — por exemplo, se você usou 8, rode com 2 e 32 (dois modelos extras, ou pelo menos um). Mantenha o restante do pipeline igual.

Refaça a auditoria da Parte 3 ($t$ em $\varepsilon=0.10$, frac $=0.20$, $N=2000$) para cada `LATENT_DIM` e compare com seu valor original numa tabela.

Em qual direção o $t$ se move quando o latente cresce? Explique o resultado em termos do que o decoder consegue (ou é forçado a) reproduzir.

*Escreva sua resposta aqui (tabela + análise).*

---

## Critérios de avaliação

| Parte | Peso | Critério |
|-------|------|----------|
| 0–1   | 20%  | VAE correto e visualizações. Q1. |
| 2     | 20%  | Marca visível e auditoria. Q2. |
| 3     | 30%  | Spread-spectrum, diagnóstico do offset, detector centralizado, teste. Q3. |
| 4     | 30%  | Ablação completa e heatmaps. Q4 com análise quantitativa. |
| 5 (bônus) | +10% | Uma das duas extensões opcionais (5.1 ou 5.2). |